# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided as a Croissant schema via this URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

This dataset contains ordered logistic regression outputs for variables affecting household adoption of indigenous and modern knowledge in rangeland management interventions in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. The Croissant schema defines the structure and sources of the data.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset package using the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print relevant dataset info
print(f"{metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset.

We'll enumerate record sets using their `@id`, and further inspect their fields.

In [ ]:
# List the record sets, their IDs, and associated fields.
print("Available record sets and their fields:")
record_set_ids = []

for record_set in metadata.record_sets:
    record_set_id = record_set.id
    record_set_ids.append(record_set_id)
    print(f"- Record set: {record_set.name} (ID: {record_set_id})")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - Field: {field.name} (ID: {field.id})")

Let's display a small sample of records for each record set by ID, as `mlcroissant` provides. This helps understand the structure and available columns.

*Note: Record set IDs are referenced via their `@id`, per standard.*

In [ ]:
# Show a sample record from each record set using its `@id` field.
for record_set_id in record_set_ids:
    print(f"\nSample records for record set {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:  # Just the first 3 for preview
            break

## 3. Data Extraction
We'll extract records from each record set and load them into pandas DataFrames for further analysis, referencing all sets and fields by their `@id`.

In [ ]:
# Extract all data from each record set using their @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for {record_set_id}:\n", df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply some data processing to the main record set(s). We'll:
- Filter records based on a specific numeric field.
- Normalize the field.
- Group by a categorical field, all via their `@id`.
- (Make sure to use IDs from output above and the Croissant schema documentation.)

In [ ]:
# For demonstration, we'll select the first record set, and search for a numeric field.
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Identify likely numeric fields by dtype and/or field listing
numeric_candidate_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if not numeric_candidate_fields:
    # If data is loaded as strings (due to CSV), try to coerce columns to numeric where possible
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors='coerce')
        if coerced.notnull().sum() > 0:
            df[col] = coerced
            numeric_candidate_fields.append(col)

if numeric_candidate_fields:
    numeric_field_id = numeric_candidate_fields[0]  # Use the first detected
    print(f"Using numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical field
    group_candidate_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
    if group_candidate_fields:
        group_field_id = group_candidate_fields[0]
        print(f"\nGrouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field and show any relationships with a group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidate_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
We explored the FAIR² Croissant dataset schema, loaded available record sets using `mlcroissant`, and performed basic processing using field and set `@id`s. From here, you can deepen analysis by joining on field IDs, inspecting value distributions, and using the referenced schema documentation to guide further investigation. This approach is flexible for any properly defined Croissant dataset.